In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Optional
import os
from pathlib import Path
import re
import json

# Set publication-ready style
plt.style.use('default')
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 1.2,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.8,
    'legend.frameon': True,
    'legend.fancybox': False,
    'legend.shadow': False,
    'legend.framealpha': 0.9,
    'legend.edgecolor': 'black',
    'legend.borderpad': 0.5,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'lines.linewidth': 2,
    'lines.markersize': 6,
})

# Set elegant color palette
colors_elegant = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#7209B7', '#2F9B5B']
sns.set_palette(colors_elegant)

# Configuration
# api_key = ""  # Fill in your wandb API key
wandb_project = "SAFE-long1k"
wandb_entity = "<WB ENTITY>"  # Leave None to use default entity

# Create output directory for PDFs
output_dir = Path("./paper_figures")
output_dir.mkdir(exist_ok=True)
print(f"Figures will be saved to: {output_dir.absolute()}")

Figures will be saved to: /Users/ihounie/repos/everysample-plots/paper_figures


In [3]:
api = wandb.Api()
# Get all runs from the project
runs = api.runs(f"{wandb_entity}/{wandb_project}" if wandb_entity else wandb_project)
# keep runs only with tag "10_ep" and config.num_train_epochs == 10
#runs = [r for r in runs if 'resilientt' in r.tags]
runs = [r for r in runs if 'tab2' in r.tags]# and 'baseline_beavertails' in r.tags]
#runs = [r for r in runs if 'dual_1000' in r.tags or 'penalty_10' in r.tags or 'dual_1000_05' in r.tags or 'dual_100' in r.tags]
# exclde loss_type penalty
#runs = [r for r in runs if r.config.get('exp', {}).get('loss_type') not in ['penalty']]
print(f"Total runs found: {len(runs)}")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /Users/ihounie/.netrc.


Total runs found: 9


In [4]:
filtered_runs=runs

### Comparison of methods

In [5]:
# Test (eval) slack CVaR95 after per-label tol adjustment — runs before the main comparison block.
# Uses the same table parsing as below; eval split = test set.

TEST_SPLIT = 'eval'

CONSTRAINT_SLACKS_PATTERN_CVaR = re.compile(
    r'^(?:.*/)?constraint_slacks_epoch_([0-9.]+)(?:_(train|eval))?_(\d+)_([A-Za-z0-9]+)\.table\.json$'
)


def _constraint_files_by_split_cvar(run):
    best_by_split_and_step = {}
    for f in run.files():
        file_name = f.name or ''
        m = CONSTRAINT_SLACKS_PATTERN_CVaR.match(file_name)
        if not m:
            continue
        epoch = float(m.group(1))
        split = (m.group(2) or 'unknown').lower()
        step = int(m.group(3))
        key = (split, step)
        prev = best_by_split_and_step.get(key)
        if prev is None:
            best_by_split_and_step[key] = (epoch, f)
        else:
            prev_epoch, prev_f = prev
            if (epoch, f.name) > (prev_epoch, prev_f.name):
                best_by_split_and_step[key] = (epoch, f)
    out = {}
    for (split, _step), (_, f) in best_by_split_and_step.items():
        out.setdefault(split, []).append(f)
    for split in out:
        out[split] = sorted(out[split], key=lambda x: x.name)
    return out


def _load_constraint_slacks_cvar(file_obj, run_id):
    run_dir = Path('./wandb_downloads') / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    local_path = run_dir / file_obj.name
    file_obj.download(root=str(run_dir), replace=True)
    with open(local_path, 'r') as f:
        data = json.load(f)

    def _to_float_or_none(v):
        try:
            return float(v)
        except Exception:
            return None

    def _to_bool_or_none(v):
        if isinstance(v, bool):
            return v
        if isinstance(v, (int, np.integer)):
            if v in (0, 1):
                return bool(v)
            return None
        if isinstance(v, str):
            s = v.strip().lower()
            if s in ('true', '1', 'yes', 'y', 't'):
                return True
            if s in ('false', '0', 'no', 'n', 'f'):
                return False
        return None

    columns = data.get('columns', []) or []
    rows = data.get('data', []) or []
    preferred_col_names = [
        'constraint_slack', 'constraint_slacks', 'slack', 'value', 'mean_nll', 'nll'
    ]
    preferred_idx = None
    safety_idx = None
    if columns:
        lower_cols = [str(c).strip().lower() for c in columns]
        for name in preferred_col_names:
            if name in lower_cols:
                preferred_idx = lower_cols.index(name)
                break
        if 'safety_label' in lower_cols:
            safety_idx = lower_cols.index('safety_label')
    table_has_safety_col = safety_idx is not None
    slacks_by_label = {True: [], False: []}
    for row in rows:
        if row is None:
            continue
        raw_safety_label = None
        picked = None
        if isinstance(row, dict):
            for k, v in row.items():
                if str(k).strip().lower() == 'safety_label':
                    raw_safety_label = v
                    break
            if preferred_idx is not None and preferred_idx < len(columns):
                key = columns[preferred_idx]
                picked = _to_float_or_none(row.get(key))
            if picked is None:
                for k, v in row.items():
                    if str(k).strip().lower() == 'safety_label':
                        continue
                    fv = _to_float_or_none(v)
                    if fv is not None:
                        picked = fv
                        break
        elif isinstance(row, (list, tuple)):
            if safety_idx is not None and safety_idx < len(row):
                raw_safety_label = row[safety_idx]
            if preferred_idx is not None and preferred_idx < len(row):
                picked = _to_float_or_none(row[preferred_idx])
            if picked is None:
                for j, v in enumerate(row):
                    if safety_idx is not None and j == safety_idx:
                        continue
                    fv = _to_float_or_none(v)
                    if fv is not None:
                        picked = fv
                        break
        else:
            picked = _to_float_or_none(row)
        safety_label = _to_bool_or_none(raw_safety_label)
        if picked is None:
            continue
        if safety_label not in (True, False):
            if table_has_safety_col:
                continue
            safety_label = True
        if safety_label is False:
            picked = -picked
        slacks_by_label[safety_label].append(picked)
    return slacks_by_label


def _apply_tol_shift(slacks_by_label, tol):
    t = 0.0 if tol is None else float(tol)
    out = {}
    for label, vals in slacks_by_label.items():
        arr = np.asarray(vals, dtype=float)
        if label is True:
            out[label] = arr - t
        else:
            # Safe: add tol, then flip sign for downstream CVaR
            out[label] = -(arr + t)
    return out


def _cvar95_upper_tail(x):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return np.nan
    p95 = np.percentile(x, 95)
    tail = x[x >= p95]
    return float(np.mean(tail)) if tail.size else float('nan')


per_run_rows = []
for run in filtered_runs:
    tol = run.config.get('exp', {}).get('tol')
    loss_type = run.config.get('exp', {}).get('loss_type')
    files_by_split = _constraint_files_by_split_cvar(run)
    files = files_by_split.get(TEST_SPLIT, [])
    if not files:
        continue
    merged = {True: [], False: []}
    for f in files:
        try:
            loaded = _load_constraint_slacks_cvar(f, run.id)
            for lb in merged:
                merged[lb].extend(loaded.get(lb, []))
        except Exception as e:
            print(f'[CVaR tol cell] skip file {f.name} for {run.name}: {e}')
    adjusted = _apply_tol_shift(merged, tol)
    arr_unsafe = adjusted.get(True)
    if arr_unsafe is None or np.asarray(arr_unsafe).size == 0:
        continue
    per_run_rows.append({
        'RunID': run.id,
        'RunName': run.name,
        'loss_type': loss_type,
        'SVaR_unsafe': _cvar95_upper_tail(arr_unsafe),
    })

per_run_df = pd.DataFrame(per_run_rows)
if per_run_df.empty:
    print('No test/eval Unsafe slack rows for CVaR95 (tol-adjusted).')
else:
    g = per_run_df.groupby('loss_type', dropna=False)['SVaR_unsafe'].agg(
        SVaR_mean='mean', SVaR_std='std', n_runs='count'
    ).reset_index()
    print('Unsafe CVaR95 (tol-adjusted test slacks): mean/std of run-level SVaR by loss_type:')
    print(g.to_string(index=False))


Unsafe CVaR95 (tol-adjusted test slacks): mean/std of run-level SVaR by loss_type:
loss_type  SVaR_mean  SVaR_std  n_runs
 aug_dual   0.092432  0.002290       3
      avg   0.312259  0.038064       3
  penalty   0.322936  0.027810       3


In [6]:
# Mean / std of wandb summary metric eval/kl_to_base_safe_mean by loss_type

KL_METRIC_KEY = 'eval/kl_to_base_safe_mean'


def _summary_float(summary, key):
    if summary is None:
        return np.nan
    try:
        v = summary.get(key) if hasattr(summary, 'get') else summary[key]
    except Exception:
        v = None
    if v is None:
        return np.nan
    try:
        f = float(v)
        return f if f == f else np.nan
    except (TypeError, ValueError):
        return np.nan


kl_rows = []
for run in filtered_runs:
    kl_rows.append({
        'RunID': run.id,
        'RunName': run.name,
        'loss_type': run.config.get('exp', {}).get('loss_type'),
        KL_METRIC_KEY: _summary_float(run.summary, KL_METRIC_KEY),
    })

kl_df = pd.DataFrame(kl_rows)
if kl_df.empty:
    print('No runs in filtered_runs.')
else:
    g = (
        kl_df.groupby('loss_type', dropna=False)[KL_METRIC_KEY]
        .agg(mean='mean', std='std', n_runs='count')
        .reset_index()
    )
    print(f'Mean and std of {KL_METRIC_KEY} by loss_type:')
    print(g.to_string(index=False))
    print()
    print('Per run:')
    print(kl_df.to_string(index=False))


Mean and std of eval/kl_to_base_safe_mean by loss_type:
loss_type     mean      std  n_runs
 aug_dual 0.186545 0.060960       3
      avg 0.387767 0.031680       3
  penalty 1.899442 0.788372       3

Per run:
   RunID                           RunName loss_type  eval/kl_to_base_safe_mean
iaf14sl4        run-alpha:5.0-tol:0.0-lr:0   penalty                   1.045147
j0trp79o run-alpha:1000.0-tol:-0.28-lr:1.0       avg                   0.408971
au48roby run-alpha:1000.0-tol:-0.08-lr:1.0  aug_dual                   0.152161
39dfpe82        run-alpha:5.0-tol:0.0-lr:0   penalty                   2.054256
mcjbu92f run-alpha:1000.0-tol:-0.28-lr:1.0       avg                   0.351350
w7cx38aj        run-alpha:5.0-tol:0.0-lr:0   penalty                   2.598922
pcstg2dh run-alpha:1000.0-tol:-0.28-lr:1.0       avg                   0.402981
snxwmzum run-alpha:1000.0-tol:-0.08-lr:1.0  aug_dual                   0.256929
cem9hkce run-alpha:1000.0-tol:-0.08-lr:1.0  aug_dual                  

In [ ]:
# <WB ENTITY>/xlam_when2call — runs tagged tab2: mean ± std by loss_type (penalty, avg, aug_dual)
WHEN2CALL_PROJECT = "<WB ENTITY>/xlam_when2call"
WHEN2CALL_METRICS = ["eval/slack_loose_cvar", "eval/objective_kl"]
# Canonical experiment keys → short row labels; row order is penalty, avg, aug_dual
LOSS_TYPE_DISPLAY = {
    "penalty_both": "penalty",
    "both_avg": "avg",
    "aug_dual_both": "aug_dual",
}
DISPLAY_ORDER = ["penalty", "avg", "aug_dual"]


def _summary_float_tc(summary, key):
    if summary is None:
        return np.nan
    try:
        v = summary.get(key) if hasattr(summary, "get") else summary[key]
    except Exception:
        v = None
    if v is None:
        return np.nan
    try:
        f = float(v)
        return f if f == f else np.nan
    except (TypeError, ValueError):
        return np.nan


def _fmt_mean_pm_std(mu, sig, n):
    if np.isnan(mu):
        return "—"
    if n <= 1 or np.isnan(sig):
        return f"{mu:g}"
    return f"{mu:g} $\\pm$ {sig:g}"


api_tc = wandb.Api()
runs_tc = api_tc.runs(WHEN2CALL_PROJECT)
runs_tc = [r for r in runs_tc if "tab2" in r.tags]
print(f"{WHEN2CALL_PROJECT}: {len(runs_tc)} run(s) with tag tab2")

rows_tc = []
for run in runs_tc:
    row = {
        "RunID": run.id,
        "RunName": run.name,
        "loss_type": run.config.get("exp", {}).get("loss_type"),
    }
    for m in WHEN2CALL_METRICS:
        v = _summary_float_tc(run.summary, m)
        if "cvar" in m and np.isfinite(v):
            v = abs(v)
        row[m] = v
    rows_tc.append(row)

df_tc = pd.DataFrame(rows_tc)
if df_tc.empty:
    print("No runs to aggregate.")
else:
    present = [lt for lt in df_tc["loss_type"].unique().tolist() if lt is not None and str(lt) != "nan"]

    def _sort_key(lt):
        label = LOSS_TYPE_DISPLAY.get(lt, lt)
        try:
            idx = DISPLAY_ORDER.index(label)
        except ValueError:
            idx = len(DISPLAY_ORDER)
        return (idx, str(label), str(lt))

    ordered_lt = sorted(present, key=_sort_key)

    summary_rows = []
    for lt in ordered_lt:
        sub = df_tc[df_tc["loss_type"] == lt]
        label = LOSS_TYPE_DISPLAY.get(lt, lt)
        row = {"loss_type": label, "n_runs": len(sub)}
        for m in WHEN2CALL_METRICS:
            vals = sub[m].dropna().astype(float)
            n = len(vals)
            mu = float(vals.mean()) if n else np.nan
            sig = float(vals.std(ddof=1)) if n > 1 else np.nan
            row[m] = _fmt_mean_pm_std(mu, sig, n)
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
    print()
    print("Per run:")
    print(df_tc.to_string(index=False))

alelab/xlam_when2call: 9 run(s) with tag tab2
loss_type  n_runs  eval/slack_loose_cvar      eval/objective_kl
  penalty       3  59.2153 $\pm$ 2.20869  12.205 $\pm$ 0.986277
      avg       3   13.2668 $\pm$ 1.2173 9.5415 $\pm$ 0.0276817
 aug_dual       3 8.30069 $\pm$ 0.864252 10.2167 $\pm$ 0.536516

Per run:
   RunID                                                       RunName     loss_type  eval/slack_loose_cvar  eval/objective_kl
4xzibf0z dpo_kl-when2call-aug_dual_both-tol:2.0-alpha:10.0-dual_lr:1.0 aug_dual_both               9.297373          10.825722
b7w4ao27      dpo_kl-when2call-both_avg-tol:2.0-alpha:10.0-dual_lr:1.0      both_avg              14.670269           9.510671
gyuiq6ps   dpo_kl-when2call-penalty_both-tol:2.0-alpha:1.0-dual_lr:0.0  penalty_both              57.163101          12.795507
n2wintp5 dpo_kl-when2call-aug_dual_both-tol:2.0-alpha:10.0-dual_lr:1.0 aug_dual_both               7.845838           9.813855
bzfmb3jk   dpo_kl-when2call-penalty_both-tol:2.0-alph

In [ ]:
# <WB ENTITY>/xlam_when2call — runs tagged tab2: mean ± std by loss_type (penalty, avg, aug_dual)
WHEN2CALL_PROJECT = "<WB ENTITY>/when2call"
WHEN2CALL_METRICS = ["eval/slack_loose_cvar", "eval/objective_kl"]
# Canonical experiment keys → short row labels; row order is penalty, avg, aug_dual
LOSS_TYPE_DISPLAY = {
    "penalty_both": "penalty",
    "both_avg": "avg",
    "aug_dual_both": "aug_dual",
}
DISPLAY_ORDER = ["penalty", "avg", "aug_dual"]


def _summary_float_tc(summary, key):
    if summary is None:
        return np.nan
    try:
        v = summary.get(key) if hasattr(summary, "get") else summary[key]
    except Exception:
        v = None
    if v is None:
        return np.nan
    try:
        f = float(v)
        return f if f == f else np.nan
    except (TypeError, ValueError):
        return np.nan


def _fmt_mean_pm_std(mu, sig, n):
    if np.isnan(mu):
        return "—"
    if n <= 1 or np.isnan(sig):
        return f"{mu:g}"
    return f"{mu:g} $\\pm$ {sig:g}"


api_tc = wandb.Api()
runs_tc = api_tc.runs(WHEN2CALL_PROJECT)
runs_tc = [r for r in runs_tc if "tb2" in r.tags]
print(f"{WHEN2CALL_PROJECT}: {len(runs_tc)} run(s) with tag tab2")

rows_tc = []
for run in runs_tc:
    row = {
        "RunID": run.id,
        "RunName": run.name,
        "loss_type": run.config.get("exp", {}).get("loss_type"),
    }
    for m in WHEN2CALL_METRICS:
        v = _summary_float_tc(run.summary, m)
        if "cvar" in m and np.isfinite(v):
            v = abs(v)
        row[m] = v
    rows_tc.append(row)

df_tc = pd.DataFrame(rows_tc)
if df_tc.empty:
    print("No runs to aggregate.")
else:
    present = [lt for lt in df_tc["loss_type"].unique().tolist() if lt is not None and str(lt) != "nan"]

    def _sort_key(lt):
        label = LOSS_TYPE_DISPLAY.get(lt, lt)
        try:
            idx = DISPLAY_ORDER.index(label)
        except ValueError:
            idx = len(DISPLAY_ORDER)
        return (idx, str(label), str(lt))

    ordered_lt = sorted(present, key=_sort_key)

    summary_rows = []
    for lt in ordered_lt:
        sub = df_tc[df_tc["loss_type"] == lt]
        label = LOSS_TYPE_DISPLAY.get(lt, lt)
        row = {"loss_type": label, "n_runs": len(sub)}
        for m in WHEN2CALL_METRICS:
            vals = sub[m].dropna().astype(float)
            n = len(vals)
            mu = float(vals.mean()) if n else np.nan
            sig = float(vals.std(ddof=1)) if n > 1 else np.nan
            row[m] = _fmt_mean_pm_std(mu, sig, n)
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))
    print()
    print("Per run:")
    print(df_tc.to_string(index=False))

alelab/when2call: 6 run(s) with tag tab2
loss_type  n_runs  eval/slack_loose_cvar       eval/objective_kl
  penalty       2   16.365 $\pm$ 4.77692 8.15545 $\pm$ 0.0430903
 aug_dual       2 9.41047 $\pm$ 0.772132  7.47373 $\pm$ 0.519266
 avg_both       2  17.9704 $\pm$ 2.20162   7.13666 $\pm$ 1.43533

Per run:
   RunID                                                       RunName     loss_type  eval/slack_loose_cvar  eval/objective_kl
khjyk37y   dpo_kl-when2call-penalty_both-tol:2.0-alpha:1.0-dual_lr:1.0  penalty_both              12.987245           8.185918
9d4xf29d      dpo_kl-when2call-avg_both-tol:2.0-alpha:10.0-dual_lr:1.0      avg_both              16.413628           8.151585
gxd5hn7w dpo_kl-when2call-aug_dual_both-tol:2.0-alpha:10.0-dual_lr:0.1 aug_dual_both               9.956450           7.106549
6lfv889r   dpo_kl-when2call-penalty_both-tol:2.0-alpha:1.0-dual_lr:1.0  penalty_both              19.742823           8.124979
z4zdr2d5 dpo_kl-when2call-aug_dual_both-tol:2.0-alpha: